# Flight Price Prediction — Consolidated Project Notebook
**IIITB Machine Learning Project** · Param Pandey (MT2026065) · Nabodaya Mete (MT2026058) · Prakhar Srivastava (MT2026071) · Varun Gupta (MT2026031)

This notebook merges the five project notebooks into one reproducible run:

| Part | Source notebook | What it does |
|---|---|---|
| A | – | Setup, data loading, exploratory data analysis (train **and** test) |
| B | `flight_preprocessing.ipynb` | Round-1 preprocessing → `1stRoundFlightdata.csv` (+ the same transform applied to the test set) |
| C | `flight_preprocessing2.ipynb` | Round-2 preprocessing → `2stRoundFlightdata.csv` (+ test set) |
| D | `flight_ML.ipynb` | Model comparison on round-1 data, blend, error analysis |
| E | `flight_ML2.ipynb` | Same comparison on round-2 data |
| F | `flight_ml3.ipynb` | Hyper-parameter tuning of XGBoost and Extra Trees |
| G | new (retrospective) | ColumnTransformer equivalent, extra engineered features, blend-weight sweep, 20 % data experiment, unsupervised analysis |
| H | – | Final model on all training rows → predictions for `test.csv` → `submission.csv` |

Cells marked **[retrospective]** were added when the notebooks were consolidated; their results were *not* part of the original Kaggle submissions.
Set `RUN_SEARCH = True` to re-run the randomized searches (≈30–40 min on 2 CPUs); by default the best settings found in the original notebooks are used.

In [1]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_predict, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer, silhouette_score
from sklearn.base import clone
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              HistGradientBoostingRegressor, VotingRegressor)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")

DATA_DIR, FIG_DIR, OUT_DIR = ".", "figures", "outputs"
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
RUN_SEARCH = False      # True = re-run all randomized searches
SEED = 42
BLUE, GREY, RED = "#2563eb", "#9ca3af", "#dc2626"

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close()

## A. Data loading and exploratory data analysis

In [2]:
train_raw = pd.read_csv(f"{DATA_DIR}/train.csv")
test_raw  = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train_raw.shape, " test:", test_raw.shape)
display(train_raw.head())
miss = pd.DataFrame({"train_missing": train_raw.isna().sum(), "test_missing": test_raw.isna().sum()})
miss["train_%"] = (miss.train_missing / len(train_raw) * 100).round(1)
miss["test_%"] = (miss.test_missing / len(test_raw) * 100).round(1)
miss.to_csv(f"{OUT_DIR}/missing_values.csv")
miss[miss.train_missing.fillna(0) + miss.test_missing.fillna(0) > 0]

train: (15589, 12)  test: (3898, 11)


,id,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,flight_class,duration,days_left,price
0,1,Wayfarer Aviation,IX 1070,Metropolis,Evening,zero,Night,Springfield,Economy,NaN,18,8909
1,2,Oceanic Airlines,6E 285,Los Santos,Night,zero,Night,Springfield,Economy,1.33,38,5115
2,3,Wayfarer Aviation,AI 2641,Night City,Evening,zero,Night,Springfield,Business,2.08,13,35247
3,4,Oceanic Airlines,6E 6704,Night City,Afternoon,zero,Evening,Metropolis,Economy,2.25,5,6457
4,5,Oceanic Airlines,6E 234,Hill Valley,Evening,one,Night,Springfield,Economy,4.83,21,6842


,train_missing,test_missing,train_%,test_%
airline,3089,799.0,19.8,20.5
duration,3095,776.0,19.9,19.9


In [3]:
# Figure: missing values in train and test
m = miss.dropna().query("train_missing > 0 or test_missing > 0")[["train_%", "test_%"]]
ax = m.plot.bar(color=[BLUE, GREY], figsize=(6, 3.5), rot=0)
ax.set_ylabel("% of rows missing"); ax.set_title("Missing values: train vs test")
for c in ax.containers: ax.bar_label(c, fmt="%.1f%%", fontsize=8)
savefig("fig_missing_values")

In [4]:
# Figure: raw vs log-transformed price (training set)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].hist(train_raw.price, bins=60, color=BLUE); ax[0].set_title("Price (Rs)"); ax[0].set_xlabel("price")
ax[1].hist(np.log1p(train_raw.price), bins=60, color=BLUE); ax[1].set_title("log1p(price)"); ax[1].set_xlabel("log1p(price)")
print("skew raw:", round(train_raw.price.skew(), 2), " skew log:", round(np.log1p(train_raw.price).skew(), 2))
savefig("fig_price_distribution")

skew raw: 2.7  skew log: 0.71


In [5]:
# Figure: price by class, stops, departure time and airline (log scale)
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
sns.boxplot(data=train_raw, x="flight_class", y="price", ax=axes[0, 0], color=BLUE, fliersize=1); axes[0, 0].set_yscale("log")
sns.boxplot(data=train_raw, x="stops", y="price", hue="flight_class", order=["zero", "one", "two_or_more"], ax=axes[0, 1], fliersize=1); axes[0, 1].set_yscale("log")
order = ["Early_Morning", "Morning", "Afternoon", "Evening", "Night", "Late_Night"]
sns.boxplot(data=train_raw, x="departure_time", y="price", hue="flight_class", order=order, ax=axes[1, 0], fliersize=1); axes[1, 0].set_yscale("log"); axes[1, 0].tick_params(axis="x", rotation=30)
a = (train_raw.assign(prefix=train_raw.flight.str[:2]).groupby("prefix")
     .agg(rows=("price", "size"), median_price=("price", "median"), business=("flight_class", lambda s: (s == "Business").mean()))
     .sort_values("rows", ascending=False))
a.median_price.plot.bar(ax=axes[1, 1], color=BLUE); axes[1, 1].set_yscale("log"); axes[1, 1].set_title("Median price by airline code (sorted by #rows)")
for ax_ in axes.flat[:3]: ax_.set_ylabel("price (Rs, log scale)")
savefig("fig_price_by_category")
a.round(2).to_csv(f"{OUT_DIR}/airline_code_profile.csv"); a.head(10)

,rows,median_price,business
prefix,,,
6E,6382,8744.5,0.263397
AI,5937,24279.0,0.520633
QP,860,7716.0,0.034884
IX,514,6779.0,0.021401
EK,443,130960.0,0.911964
UL,422,71131.5,0.834123
EY,407,134670.0,0.889435
TG,228,102184.5,0.934211
SG,147,7333.0,0.000000


In [6]:
# Figure: days_left and duration vs price, by class
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
g = train_raw.groupby(["days_left", "flight_class"]).price.median().unstack()
ax[0].plot(g.index, g["Economy"], color=GREY, label="Economy"); ax[0].plot(g.index, g["Business"], color=BLUE, label="Business")
ax[0].set_yscale("log"); ax[0].set_xlabel("days left before departure"); ax[0].set_ylabel("median price (Rs, log)"); ax[0].legend(); ax[0].set_title("Median fare by days left: highest 1-3 days before departure")
for cls, col in [("Economy", GREY), ("Business", BLUE)]:
    s = train_raw[train_raw.flight_class == cls]
    ax[1].scatter(s.duration, s.price, s=3, alpha=0.3, color=col, label=cls)
ax[1].set_yscale("log"); ax[1].set_xlabel("duration (h)"); ax[1].set_ylabel("price (Rs, log)"); ax[1].legend(markerscale=4); ax[1].set_title("Duration vs price")
savefig("fig_days_duration_vs_price")
print("corr(duration, price) =", round(train_raw.duration.corr(train_raw.price), 3))
print("corr(days_left, price) =", round(train_raw.days_left.corr(train_raw.price), 3))

corr(duration, price) = 0.699
corr(days_left, price) = -0.023


In [7]:
# Figure: train vs test distributions (is the test set like the training set?)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax_, col in zip(axes[:2], ["days_left", "duration"]):
    ax_.hist(train_raw[col].dropna(), bins=40, density=True, alpha=0.6, color=BLUE, label="train")
    ax_.hist(test_raw[col].dropna(), bins=40, density=True, alpha=0.6, color=GREY, label="test"); ax_.set_title(col); ax_.legend()
share = pd.DataFrame({"train": train_raw.flight_class.value_counts(normalize=True), "test": test_raw.flight_class.value_counts(normalize=True)})
share.plot.bar(ax=axes[2], color=[BLUE, GREY], rot=0); axes[2].set_title("class share")
savefig("fig_train_vs_test")
share.round(3)

,train,test
flight_class,,
Economy,0.591,0.6
Business,0.409,0.4


In [8]:
# Correlation heatmap of numeric / ordinal features
num = train_raw.assign(stops=train_raw.stops.map({"zero": 0, "one": 1, "two_or_more": 2}),
                       business=(train_raw.flight_class == "Business").astype(int),
                       log_price=np.log1p(train_raw.price))[["id", "stops", "business", "duration", "days_left", "log_price"]]
plt.figure(figsize=(6, 4.5)); sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Correlation matrix (train)"); savefig("fig_correlation")

## B. Preprocessing round 1 (`flight_preprocessing.ipynb`)
Steps, in the original order: airline filled from the flight-code prefix → duration filled with hierarchical group medians → `id` and `airline` dropped → flight split into prefix + number → one-hot / ordinal encoding → log1p(price) → directional `route` (source × destination) → `flight_number` dropped → duplicates dropped.
All look-up tables (prefix→airline, duration medians) are learned on the **training set only** and re-used for the test set.

In [9]:
df = train_raw.copy()
df["prefix"] = df["flight"].str.strip().str.upper().str[:2]
known = df[df.airline.notna()]
per_prefix = known.groupby("prefix")["airline"].agg(n_airlines="nunique", top=lambda s: s.mode().iloc[0],
                                                    purity=lambda s: s.value_counts(normalize=True).iloc[0])
print("prefixes mapping to >1 airline:", (per_prefix.n_airlines > 1).sum(), "| min purity:", per_prefix.purity.min())
prefix_to_airline = per_prefix["top"].to_dict()
missing = df.airline.isna(); filled = df.loc[missing, "prefix"].map(prefix_to_airline)
print("Fillable:", filled.notna().sum(), "of", missing.sum())
df.loc[missing, "airline"] = filled

prefixes mapping to >1 airline: 0 | min purity: 1.0
Fillable: 3089 of 3089


In [10]:
# Duration imputation: validate group-median approach on an 80/20 split of known durations
data = df.dropna(subset=["duration"]).copy()
print("Duration-price correlation:", round(data.duration.corr(data.price), 4))
rng = np.random.default_rng(42); idx = rng.permutation(data.index); cut = int(0.8 * len(idx))
reference, validation = data.loc[idx[:cut]], data.loc[idx[cut:]]
group_cols = ["flight", "source_city", "destination_city", "stops"]
gm = reference.groupby(group_cols)["duration"].median()
pred_group = pd.Series(pd.MultiIndex.from_frame(validation[group_cols]).map(gm), index=validation.index)
mask = pred_group.notna(); err = validation.loc[mask, "duration"] - pred_group[mask]
print(f"Coverage: {mask.mean()*100:.2f}% | MAE: {err.abs().mean():.3f} h | RMSE: {np.sqrt((err**2).mean()):.3f} h")

Duration-price correlation: 0.6994
Coverage: 95.76% | MAE: 0.188 h | RMSE: 1.383 h


In [11]:
KNOWN_DUR = train_raw.dropna(subset=["duration"])
LEVELS = [["flight", "source_city", "destination_city", "stops"],
          ["source_city", "destination_city", "stops"],
          ["source_city", "destination_city"]]
DUR_MEDIANS = [KNOWN_DUR.groupby(c)["duration"].median() for c in LEVELS]
GLOBAL_MED = KNOWN_DUR.duration.median()

def fill_duration_round1(d):
    d = d.copy()
    for cols, med in zip(LEVELS, DUR_MEDIANS):
        fill = pd.Series(pd.MultiIndex.from_frame(d[cols]).map(med), index=d.index)
        miss = d.duration.isna(); d.loc[miss, "duration"] = fill[miss]
    d["duration"] = d["duration"].fillna(GLOBAL_MED)
    return d

def round1_transform(raw, is_train):
    d = raw.copy()
    d["prefix"] = d["flight"].str.strip().str.upper().str[:2]
    d = fill_duration_round1(d)
    d = d.drop(columns=["id", "airline"])
    d["flight_number"] = d["flight"].str[2:].str.strip()
    d = d.drop(columns=["flight"])
    d = pd.get_dummies(d, columns=["source_city"], dtype=int)
    d = pd.get_dummies(d, columns=["departure_time"], dtype=int)
    d = pd.get_dummies(d, columns=["arrival_time"], dtype=int)
    d["stops"] = d["stops"].map({"zero": 0, "one": 1, "two_or_more": 2})
    d = pd.get_dummies(d, columns=["destination_city"], dtype=int)
    d["flight_class"] = d["flight_class"].map({"Economy": 0, "Business": 1})
    d = pd.get_dummies(d, columns=["prefix"], dtype=int)
    if is_train: d["price"] = np.log1p(d["price"])
    sc, dc = d.filter(regex=r"^source_city_").columns, d.filter(regex=r"^destination_city_").columns
    d["route"] = (d[sc].idxmax(axis=1).str.replace("source_city_", "", regex=False) + "_to_" +
                  d[dc].idxmax(axis=1).str.replace("destination_city_", "", regex=False))
    d = d.drop(columns=list(sc) + list(dc))
    d = pd.get_dummies(d, columns=["route"], dtype=int)
    d = d.drop(columns=["flight_number"])
    if is_train: d = d.drop_duplicates().reset_index(drop=True)
    return d

print("missing durations before:", train_raw.duration.isna().sum(), "train /", test_raw.duration.isna().sum(), "test")
df1 = round1_transform(train_raw, True)
X1_full, y1_full = df1.drop(columns=["price"]), df1["price"]
X1_kaggle = round1_transform(test_raw, False).reindex(columns=X1_full.columns, fill_value=0)
print("round-1 train:", df1.shape, "| test features:", X1_kaggle.shape)
df1.to_csv("1stRoundFlightdata.csv", index=False)

missing durations before: 3095 train / 776 test
round-1 train: (14431, 66) | test features: (3898, 65)


## C. Preprocessing round 2 (`flight_preprocessing2.ipynb`)
Changes vs round 1: missing durations predicted by a model (best of six, by 5-fold out-of-fold MAE) instead of group medians; airline codes with fewer than 50 rows grouped into `flight_code_other`.

In [12]:
def round2_base(raw):
    d = raw.drop(columns=["id", "airline"]).copy()
    d["flight_code"] = d["flight"].str.strip().str.upper().str[:2]
    d["flight_number"] = d["flight"].str[2:].str.strip().astype(int)
    d = d.drop(columns=["flight"])
    d["stops"] = d["stops"].map({"zero": 0, "one": 1, "two_or_more": 2})
    d["flight_class"] = d["flight_class"].map({"Economy": 0, "Business": 1})
    return d

d2 = round2_base(train_raw); t2 = round2_base(test_raw)
known2 = d2.dropna(subset=["duration"]).copy()
cat_cols = ["flight_code", "source_city", "destination_city", "departure_time", "arrival_time"]
num_cols = ["stops", "flight_class", "flight_number"]
Xd = pd.get_dummies(known2[cat_cols + num_cols], columns=cat_cols, dtype=int); yd = known2["duration"]

dur_models = {
    "Random Forest": RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42),
    "Extra Trees": ExtraTreesRegressor(n_estimators=300, n_jobs=-1, random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=8, n_jobs=-1, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=63, min_child_samples=5, random_state=42, verbose=-1),
    "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, depth=8, random_state=42, verbose=0),
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
one_stop = (known2.stops == 1).values; rows = []
for name, m in dur_models.items():
    p = cross_val_predict(m, Xd, yd, cv=cv); e = np.abs(p - yd.values)
    rows.append({"model": name, "MAE (h)": e.mean(), "RMSE (h)": np.sqrt(mean_squared_error(yd, p)),
                 "one-stop MAE (h)": e[one_stop].mean(), "% off by >1h": (e > 1).mean() * 100})
dur_results = pd.DataFrame(rows).set_index("model").sort_values("MAE (h)").round(3)
dur_results.to_csv(f"{OUT_DIR}/round2_duration_imputation.csv"); dur_results

,MAE (h),RMSE (h),one-stop MAE (h),% off by >1h
model,,,,
Extra Trees,0.108,0.845,0.332,2.297
Random Forest,0.125,0.804,0.379,2.577
XGBoost,0.134,0.823,0.380,2.169
LightGBM,0.148,0.815,0.398,2.481
Hist Gradient Boosting,0.224,0.860,0.572,3.882
CatBoost,0.229,0.779,0.555,3.546


In [13]:
best_dur = dur_models[dur_results.index[0]]; print("duration model:", dur_results.index[0])
best_dur.fit(Xd, yd)
for d in (d2, t2):
    miss = d.duration.isna()
    Xm = pd.get_dummies(d.loc[miss, cat_cols + num_cols], columns=cat_cols, dtype=int).reindex(columns=Xd.columns, fill_value=0)
    d.loc[miss, "duration"] = best_dur.predict(Xm)

def round2_encode(d):
    d = d.copy()
    d["route"] = d["source_city"] + "_to_" + d["destination_city"]
    d = d.drop(columns=["source_city", "destination_city", "flight_number"])
    return pd.get_dummies(d, columns=["route", "departure_time", "arrival_time", "flight_code"], dtype=int)

d2e = round2_encode(d2); print("duplicate rows:", d2e.duplicated().sum())
d2e = d2e.drop_duplicates().reset_index(drop=True)
code_cols = [c for c in d2e.columns if c.startswith("flight_code_")]
counts = d2e[code_cols].sum().sort_values(); rare = counts[counts < 50].index.tolist()
print("grouped into other:", {r.replace('flight_code_', ''): int(counts[r]) for r in rare})
d2e["flight_code_other"] = d2e[rare].sum(axis=1); d2e = d2e.drop(columns=rare)
d2e["price"] = np.log1p(d2e["price"])
X2_full, y2_full = d2e.drop(columns=["price"]), d2e["price"]

t2e = round2_encode(t2)
t_codes = [c for c in t2e.columns if c.startswith("flight_code_")]
kept = [c for c in X2_full.columns if c.startswith("flight_code_") and c != "flight_code_other"]
t2e["flight_code_other"] = t2e[[c for c in t_codes if c not in kept]].sum(axis=1)
X2_kaggle = t2e.reindex(columns=X2_full.columns, fill_value=0)
print("round-2 train:", d2e.shape, "| test features:", X2_kaggle.shape,
      "| test rows in 'other':", int(X2_kaggle.flight_code_other.sum()))
d2e.to_csv("2stRoundFlightdata.csv", index=False)

duration model: Extra Trees


duplicate rows: 892
grouped into other: {'9I': 2, 'RA': 2, 'S5': 3, 'KU': 7, 'SQ': 8, 'FZ': 10, 'WY': 29, 'GF': 31}
round-2 train: (14697, 59) | test features: (3898, 58) | test rows in 'other': 29


In [14]:
# Figure: rows per airline code with the grouping cut-off
c = pd.concat([counts]).sort_values(ascending=False); c.index = c.index.str.replace("flight_code_", "")
plt.figure(figsize=(10, 3.5)); plt.bar(c.index, c.values, color=[GREY if v < 50 else BLUE for v in c.values])
plt.yscale("log"); plt.axhline(50, color=RED, ls="--", lw=1); plt.text(len(c) - 0.5, 60, "cut-off = 50 rows", ha="right", color=RED, fontsize=9)
plt.ylabel("rows (log scale)"); plt.title("Rows per airline code (round-2 data); grey codes grouped into 'other'")
savefig("fig_rare_codes")

## D. Supervised models on round-1 data (`flight_ML.ipynb`)
Hold-out: 20 % stratified by `flight_class`, `random_state=42`. All metrics are computed on log1p(price) (log RMSE, log MAE, R²) and on rupees (MAE, MAPE).

In [15]:
def metrics(y_true, y_pred):
    t, p = np.expm1(np.asarray(y_true)), np.expm1(np.asarray(y_pred))
    return {"log RMSE": np.sqrt(mean_squared_error(y_true, y_pred)), "log MAE": mean_absolute_error(y_true, y_pred),
            "R2 (log)": r2_score(y_true, y_pred), "R2 (Rs)": r2_score(t, p),
            "MAE (Rs)": mean_absolute_error(t, p), "MAPE (%)": np.mean(np.abs(t - p) / t) * 100}

def cv_metrics(model, X, y, cv):
    rows = []
    for tr, va in cv.split(X):
        m = clone(model).fit(X.iloc[tr], y.iloc[tr]); rows.append(metrics(y.iloc[va], m.predict(X.iloc[va])))
    f = pd.DataFrame(rows); out = f.mean().to_dict()
    out["log RMSE sd"] = f["log RMSE"].std(); out["MAPE sd"] = f["MAPE (%)"].std(); return out

def make_models(hgb_params):
    return {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42),
        "Gradient Boosting (HGB)": HistGradientBoostingRegressor(max_iter=500, learning_rate=0.1, random_state=42),
        "Gradient Boosting (tuned)": HistGradientBoostingRegressor(random_state=42, **hgb_params),
        "Extra Trees": ExtraTreesRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42),
        "LightGBM": LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=63, min_child_samples=10, reg_lambda=0.1, random_state=42, verbose=-1),
        "XGBoost": XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=8, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42),
        "CatBoost": CatBoostRegressor(iterations=800, learning_rate=0.05, depth=8, random_state=42, verbose=0),
    }

def tune_hgb(X, y, recorded):
    if not RUN_SEARCH: return recorded
    grid = {"learning_rate": [0.03, 0.05, 0.1, 0.15], "max_iter": [300, 500, 800], "max_leaf_nodes": [15, 31, 63],
            "min_samples_leaf": [10, 20, 40], "l2_regularization": [0.0, 0.1, 1.0]}
    s = RandomizedSearchCV(HistGradientBoostingRegressor(random_state=42), grid, n_iter=20, cv=5,
                           scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42).fit(X, y)
    print("best HGB:", s.best_params_, "CV RMSE", -s.best_score_); return s.best_params_

def run_round(X, y, hgb_recorded, tag):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=X["flight_class"])
    print(tag, "train", Xtr.shape, "holdout", Xte.shape)
    cm = ytr.groupby(Xtr.flight_class).mean()
    hold = [{"model": "Baseline (class mean)", **metrics(yte, Xte.flight_class.map(cm))}]
    models = make_models(tune_hgb(Xtr, ytr, hgb_recorded)); fitted = {}
    for name, m in models.items():
        t0 = time.time(); fitted[name] = m.fit(Xtr, ytr)
        hold.append({"model": name, **metrics(yte, m.predict(Xte)), "fit s": round(time.time() - t0, 1)})
    hold = pd.DataFrame(hold).set_index("model")
    cvk = KFold(n_splits=5, shuffle=True, random_state=42); cvr = []
    cv_set = {k: v for k, v in models.items() if k != "Gradient Boosting (HGB)"}
    cv_set["Voting (XGB + ET)"] = VotingRegressor([("xgb", models["XGBoost"]), ("et", models["Extra Trees"])])
    cv_set["Voting (XGB + ET + HGB)"] = VotingRegressor([("xgb", models["XGBoost"]), ("et", models["Extra Trees"]), ("hgb", models["Gradient Boosting (tuned)"])])
    for name, m in cv_set.items():
        cvr.append({"model": name, **cv_metrics(m, Xtr, ytr, cvk)})
    cvr = pd.DataFrame(cvr).set_index("model").sort_values("log RMSE")
    hold.round(4).to_csv(f"{OUT_DIR}/{tag}_holdout.csv"); cvr.round(4).to_csv(f"{OUT_DIR}/{tag}_cv.csv")
    return dict(Xtr=Xtr, Xte=Xte, ytr=ytr, yte=yte, models=models, fitted=fitted, hold=hold, cv=cvr)

HGB_R1 = {"min_samples_leaf": 10, "max_leaf_nodes": 63, "max_iter": 800, "learning_rate": 0.03, "l2_regularization": 0.1}
R1 = run_round(X1_full, y1_full, HGB_R1, "round1")
display(R1["hold"].round(4)); display(R1["cv"].round(4))

round1 train (11544, 65) holdout (2887, 65)


,log RMSE,log MAE,R2 (log),R2 (Rs),MAE (Rs),MAPE (%),fit s
model,,,,,,,
Baseline (class mean),0.4921,0.3603,0.7642,0.3553,15379.9947,34.6792,NaN
Ridge,0.2494,0.1756,0.9394,0.8420,6552.2884,17.2647,0.0
Random Forest,0.1495,0.0839,0.9782,0.9204,3787.1637,8.4235,6.0
Gradient Boosting (HGB),0.1539,0.0970,0.9769,0.9186,4166.3914,9.6808,1.3
Gradient Boosting (tuned),0.1498,0.0919,0.9781,0.9246,3957.7951,9.2079,4.2
Extra Trees,0.1418,0.0817,0.9804,0.9212,3692.2862,8.0809,7.9
LightGBM,0.1472,0.0886,0.9789,0.9217,3944.4513,8.8604,1.1
XGBoost,0.1367,0.0827,0.9818,0.9256,3708.0720,8.2155,1.6
CatBoost,0.1441,0.0927,0.9798,0.9233,3951.7212,9.1130,1.7


,log RMSE,log MAE,R2 (log),R2 (Rs),MAE (Rs),MAPE (%),log RMSE sd,MAPE sd
model,,,,,,,,
Voting (XGB + ET + HGB),0.1349,0.0810,0.9818,0.9351,3376.1168,8.0072,0.0051,0.2733
Voting (XGB + ET),0.1352,0.0802,0.9817,0.9343,3341.3528,7.9271,0.0054,0.2756
XGBoost,0.1366,0.0832,0.9813,0.9330,3449.2567,8.2573,0.0044,0.2977
Extra Trees,0.1417,0.0819,0.9799,0.9302,3411.4245,8.0997,0.0065,0.2651
Gradient Boosting (tuned),0.1418,0.0872,0.9798,0.9298,3623.4852,8.6721,0.0051,0.3306
LightGBM,0.1425,0.0873,0.9797,0.9290,3636.8489,8.6767,0.0041,0.3121
Random Forest,0.1426,0.0834,0.9796,0.9282,3515.6440,8.3364,0.0053,0.2464
CatBoost,0.1456,0.0941,0.9788,0.9313,3734.8921,9.3008,0.0039,0.2610
Ridge,0.2503,0.1767,0.9372,0.8368,6379.1638,17.3876,0.0060,0.2680


In [16]:
# Figure: 5-fold CV comparison (round 1)
c = R1["cv"].sort_values("log RMSE", ascending=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
cols = [BLUE if "Voting (XGB + ET)" == i else GREY for i in c.index]
ax[0].barh(c.index, c["log RMSE"], xerr=c["log RMSE sd"], color=cols); ax[0].set_xlabel("CV log RMSE (lower is better)")
ax[1].barh(c.index, c["MAPE (%)"], xerr=c["MAPE sd"], color=cols); ax[1].set_xlabel("CV MAPE % (lower is better)"); ax[1].set_yticklabels([])
fig.suptitle("Round-1 data: 5-fold cross-validation on the training split")
savefig("fig_model_comparison_round1")

### D.2 Error analysis of the XGB + ET blend (out-of-fold on the training split)

In [17]:
Xtr, ytr = R1["Xtr"], R1["ytr"]
cvk = KFold(n_splits=5, shuffle=True, random_state=42)
oof_xgb = cross_val_predict(R1["models"]["XGBoost"], Xtr, ytr, cv=cvk)
oof_et  = cross_val_predict(R1["models"]["Extra Trees"], Xtr, ytr, cv=cvk)
oof_blend = (oof_xgb + oof_et) / 2          # VotingRegressor = unweighted mean
print({k: round(v, 4) for k, v in metrics(ytr, oof_blend).items()})
errors = Xtr.copy(); errors["actual_rs"] = np.expm1(ytr); errors["pred_rs"] = np.expm1(oof_blend)
errors["error_pct"] = (errors.pred_rs - errors.actual_rs).abs() / errors.actual_rs * 100
def label(prefix):
    cols = [c for c in errors.columns if c.startswith(prefix)]
    return errors[cols].idxmax(axis=1).str.replace(prefix, "", regex=False)
errors["route"] = label("route_"); errors["airline_prefix"] = label("prefix_")
errors["class"] = errors.flight_class.map({0: "Economy", 1: "Business"})
errors["class_stops"] = errors["class"] + " / " + errors.stops.astype(str) + " stops"
errors["days_bucket"] = pd.cut(errors.days_left, [0, 3, 7, 14, 30, 60], labels=["1-3", "4-7", "8-14", "15-30", "31+"])
print(errors.error_pct.describe().round(1))
def summarize(col):
    g = errors.groupby(col, observed=True).error_pct.agg(rows="count", mean_err="mean", median_err="median")
    g["share_of_rows_%"] = g.rows / len(errors) * 100
    g["share_of_error_%"] = errors.groupby(col, observed=True).error_pct.sum() / errors.error_pct.sum() * 100
    return g.round(1).sort_values("mean_err", ascending=False)
for col in ["class", "class_stops", "days_bucket", "airline_prefix"]:
    s = summarize(col); s.to_csv(f"{OUT_DIR}/error_by_{col}.csv"); display(s.head(8))

{'log RMSE': np.float64(0.1353), 'log MAE': 0.0802, 'R2 (log)': 0.9817, 'R2 (Rs)': 0.9341, 'MAE (Rs)': 3341.3683, 'MAPE (%)': np.float64(7.9271)}
count    11544.0
mean         7.9
std         10.6
min          0.0
25%          1.6
50%          4.2
75%         10.0
max        156.7
Name: error_pct, dtype: float64


,rows,mean_err,median_err,share_of_rows_%,share_of_error_%
class,,,,,
Business,4764,9.8,5.7,41.3,50.9
Economy,6780,6.6,3.5,58.7,49.1


,rows,mean_err,median_err,share_of_rows_%,share_of_error_%
class_stops,,,,,
Economy / 2 stops,1,39.3,39.3,0.0,0.0
Business / 0 stops,2055,10.2,7.0,17.8,23.0
Business / 1 stops,2658,9.5,4.8,23.0,27.5
Business / 2 stops,51,7.5,5.8,0.4,0.4
Economy / 1 stops,795,7.4,3.4,6.9,6.4
Economy / 0 stops,5984,6.5,3.5,51.8,42.6


,rows,mean_err,median_err,share_of_rows_%,share_of_error_%
days_bucket,,,,,
1-3,716,14.2,8.5,6.2,11.1
4-7,1021,8.4,4.8,8.8,9.3
8-14,2103,8.2,4.5,18.2,18.9
15-30,4284,7.4,3.9,37.1,34.7
31+,3420,6.9,3.7,29.6,25.9


,rows,mean_err,median_err,share_of_rows_%,share_of_error_%
airline_prefix,,,,,
S5,2,56.6,56.6,0.0,0.1
RA,2,31.3,31.3,0.0,0.1
EY,300,24.0,17.2,2.6,7.9
MH,71,15.7,10.5,0.6,1.2
SG,123,15.4,7.6,1.1,2.1
WY,25,12.0,7.6,0.2,0.3
TG,180,11.2,10.2,1.6,2.2
UL,311,10.7,5.7,2.7,3.6


In [18]:
# Figure: actual vs predicted (OOF) and error by group
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for cls, col in [("Economy", GREY), ("Business", BLUE)]:
    s = errors[errors["class"] == cls]; axes[0].scatter(s.actual_rs, s.pred_rs, s=3, alpha=0.3, color=col, label=cls)
lim = [3e3, 3.5e5]; axes[0].plot(lim, lim, color=RED, lw=1); axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("actual price (Rs)"); axes[0].set_ylabel("predicted (Rs)"); axes[0].legend(markerscale=4); axes[0].set_title("Out-of-fold: actual vs predicted")
overall = errors.error_pct.mean()
for ax_, col, title in [(axes[1], "days_bucket", "Mean error by days left"), (axes[2], "airline_prefix", "Mean error by airline code (n>=20)")]:
    g = errors.groupby(col, observed=True).error_pct.agg(["mean", "count"]).query("count >= 20").sort_values("mean")
    ax_.barh(g.index.astype(str), g["mean"], color=[RED if m > overall else GREY for m in g["mean"]])
    ax_.axvline(overall, color=RED, ls="--", lw=1); ax_.set_xlabel("mean absolute % error"); ax_.set_title(title)
savefig("fig_error_analysis")

In [19]:
# EY deep dive + irreducible-noise check (rows with identical features)
feat = list(Xtr.columns)
ey = errors[errors.airline_prefix == "EY"]
print("EY rows:", len(ey), "| business share:", round(ey.flight_class.mean(), 2), "| mean stops:", round(ey.stops.mean(), 2),
      "| mean duration:", round(ey.duration.mean(), 1), "| mean price:", round(ey.actual_rs.mean()), "| mean err %:", round(ey.error_pct.mean(), 1))
dup = errors.assign(lp=np.log1p(errors.actual_rs)).groupby(feat).lp.agg(["std", "count"]).query("count > 1")
dup = dup.reset_index()[["flight_class", "std", "count"]]
print("log-price std among rows with identical features:", dup.groupby("flight_class")["std"].mean().round(3).to_dict())

EY rows: 300 | business share: 0.91 | mean stops: 1.0 | mean duration: 13.8 | mean price: 140049 | mean err %: 24.0
log-price std among rows with identical features: {0: 0.058, 1: 0.131}


In [20]:
# Separate models per class vs one joint blend (OOF on training split)
sep = np.zeros(len(ytr))
for cls in [0, 1]:
    msk = (Xtr.flight_class == cls).values
    b = VotingRegressor([("xgb", R1["models"]["XGBoost"]), ("et", R1["models"]["Extra Trees"])])
    sep[msk] = cross_val_predict(b, Xtr[msk], ytr[msk], cv=cvk)
comp = pd.DataFrame({"joint blend": metrics(ytr, oof_blend), "separate per class": metrics(ytr, sep)}).T
comp.round(4).to_csv(f"{OUT_DIR}/separate_vs_joint.csv"); comp.round(4)

,log RMSE,log MAE,R2 (log),R2 (Rs),MAE (Rs),MAPE (%)
joint blend,0.1353,0.0802,0.9817,0.9341,3341.3683,7.9271
separate per class,0.1372,0.0803,0.9811,0.9278,3416.1376,7.9887


## E. Supervised models on round-2 data (`flight_ML2.ipynb`)

In [21]:
HGB_R2 = HGB_R1      # flight_ML2 re-ran the same search; recorded CV RMSE of the best setting: 0.1397
R2 = run_round(X2_full, y2_full, HGB_R2, "round2")
display(R2["hold"].round(4)); display(R2["cv"].round(4))

round2 train (11757, 58) holdout (2940, 58)


,log RMSE,log MAE,R2 (log),R2 (Rs),MAE (Rs),MAPE (%),fit s
model,,,,,,,
Baseline (class mean),0.4703,0.3464,0.7767,0.3781,14168.2735,34.2136,NaN
Ridge,0.2504,0.1754,0.9367,0.8376,6289.9940,17.3779,0.0
Random Forest,0.1469,0.0825,0.9782,0.9184,3581.8194,8.0955,6.9
Gradient Boosting (HGB),0.1533,0.0957,0.9763,0.9147,3955.7144,9.4298,1.2
Gradient Boosting (tuned),0.1487,0.0897,0.9777,0.9142,3771.2589,8.8149,3.9
Extra Trees,0.1423,0.0797,0.9796,0.9182,3453.4702,7.8263,8.4
LightGBM,0.1460,0.0867,0.9785,0.9168,3695.3207,8.5217,1.1
XGBoost,0.1387,0.0818,0.9806,0.9193,3477.2226,8.0638,1.6
CatBoost,0.1453,0.0916,0.9787,0.9213,3727.1165,8.9890,1.7


,log RMSE,log MAE,R2 (log),R2 (Rs),MAE (Rs),MAPE (%),log RMSE sd,MAPE sd
model,,,,,,,,
Voting (XGB + ET + HGB),0.1324,0.0794,0.9825,0.9347,3356.4682,7.8688,0.0039,0.1882
Voting (XGB + ET),0.1328,0.0784,0.9824,0.9342,3325.1228,7.7709,0.0048,0.2347
XGBoost,0.1333,0.0812,0.9823,0.9336,3390.2315,8.0707,0.0042,0.2334
Gradient Boosting (tuned),0.1390,0.0860,0.9807,0.9301,3595.9273,8.5473,0.0025,0.1090
Extra Trees,0.1399,0.0801,0.9804,0.9295,3420.0747,7.9650,0.0053,0.3075
LightGBM,0.1404,0.0862,0.9803,0.9290,3612.4308,8.5767,0.0028,0.1427
Random Forest,0.1404,0.0821,0.9803,0.9293,3479.1000,8.1661,0.0040,0.1439
CatBoost,0.1418,0.0916,0.9799,0.9315,3693.8291,9.0564,0.0037,0.0949
Ridge,0.2534,0.1779,0.9359,0.8255,6584.5223,17.5737,0.0038,0.2907


In [22]:
# Figure: round 1 vs round 2 (CV log RMSE on each round's own training split)
cmp = pd.DataFrame({"Round 1": R1["cv"]["log RMSE"], "Round 2": R2["cv"]["log RMSE"]}).sort_values("Round 1", ascending=False)
ax = cmp.plot.barh(color=[BLUE, GREY], figsize=(8, 5)); ax.set_xlabel("CV log RMSE"); ax.set_xlim(0.12, cmp.max().max() * 1.02)
ax.set_title("Round 1 vs round 2 preprocessing (5-fold CV)"); savefig("fig_round1_vs_round2")
cmp.round(4)

,Round 1,Round 2
model,,
Ridge,0.2503,0.2534
CatBoost,0.1456,0.1418
Random Forest,0.1426,0.1404
LightGBM,0.1425,0.1404
Gradient Boosting (tuned),0.1418,0.1390
Extra Trees,0.1417,0.1399
XGBoost,0.1366,0.1333
Voting (XGB + ET),0.1352,0.1328
Voting (XGB + ET + HGB),0.1349,0.1324


In [23]:
# Kaggle public-leaderboard scores (recorded from the submissions, not recomputed)
kaggle = pd.DataFrame([
    ("Round 1", "Random Forest", 0.95467), ("Round 1", "XGBoost", 0.95978), ("Round 1", "Extra Trees", 0.96051),
    ("Round 1", "Blend XGB + ET", 0.96238), ("Round 1", "Per-class blend", 0.96051),
    ("Round 2", "Random Forest", 0.95157), ("Round 2", "Hist Gradient Boosting", 0.95496), ("Round 2", "XGBoost", 0.95798),
    ("Round 2", "Extra Trees", 0.95905), ("Round 2", "Blend XGB + ET", 0.96077)], columns=["data", "model", "public score"])
kaggle.to_csv(f"{OUT_DIR}/kaggle_scores.csv", index=False)
k = kaggle.sort_values("public score")
plt.figure(figsize=(8, 4.2)); plt.barh(k.data + " · " + k.model, k["public score"], color=[BLUE if s == k["public score"].max() else GREY for s in k["public score"]])
plt.xlim(0.945, 0.965); plt.xlabel("Kaggle public score"); plt.title("Kaggle submissions: best = round-1 XGB + ET blend (0.96238)")
for i, s in enumerate(k["public score"]): plt.text(s + 0.0003, i, f"{s:.5f}", va="center", fontsize=8)
savefig("fig_kaggle_scores"); kaggle

,data,model,public score
0,Round 1,Random Forest,0.95467
1,Round 1,XGBoost,0.95978
2,Round 1,Extra Trees,0.96051
3,Round 1,Blend XGB + ET,0.96238
4,Round 1,Per-class blend,0.96051
5,Round 2,Random Forest,0.95157
6,Round 2,Hist Gradient Boosting,0.95496
7,Round 2,XGBoost,0.95798
8,Round 2,Extra Trees,0.95905
9,Round 2,Blend XGB + ET,0.96077


## F. Hyper-parameter tuning (`flight_ml3.ipynb`)
Two rounds of randomized search on the round-1 training split: first scored on log RMSE, then on R² of the raw price (the Kaggle metric).
The final comparison uses untuned *default* models (300 trees) as the reference, exactly as in `flight_ml3`.

In [24]:
def raw_price_r2(y_true_log, y_pred_log): return r2_score(np.expm1(y_true_log), np.expm1(y_pred_log))
kaggle_scorer = make_scorer(raw_price_r2)
from scipy.stats import randint, uniform
XGB_TUNED = {"colsample_bytree": 0.5852620618436457, "gamma": 0.03252579649263976, "learning_rate": 0.1433551198429333,
             "max_depth": 4, "min_child_weight": 2, "n_estimators": 400, "reg_alpha": 0.015966252220214194,
             "reg_lambda": 1.254469128110745, "subsample": 0.6964101864104046}
ET_TUNED = {"n_estimators": 700, "min_samples_split": 6, "min_samples_leaf": 1, "max_features": 0.85, "max_depth": 30, "bootstrap": True}
if RUN_SEARCH:
    xs = RandomizedSearchCV(XGBRegressor(random_state=42, n_jobs=1, verbosity=0),
        {"n_estimators": [400, 600, 800, 1000, 1200, 1500], "max_depth": randint(3, 12), "learning_rate": uniform(0.02, 0.13),
         "min_child_weight": randint(1, 9), "subsample": uniform(0.6, 0.4), "colsample_bytree": uniform(0.5, 0.5),
         "gamma": uniform(0.0, 0.5), "reg_lambda": uniform(0.1, 5.0), "reg_alpha": uniform(0.0, 1.0)},
        n_iter=30, scoring=kaggle_scorer, cv=cvk, n_jobs=-1, random_state=42).fit(R1["Xtr"], R1["ytr"])
    es = RandomizedSearchCV(ExtraTreesRegressor(random_state=42, n_jobs=1),
        {"n_estimators": [300, 500, 700, 1000], "max_depth": [None, 15, 20, 30, 40], "min_samples_leaf": [1, 2, 3, 4, 5],
         "min_samples_split": [2, 4, 6, 10], "max_features": [0.5, 0.7, 0.85, 1.0], "bootstrap": [True, False]},
        n_iter=20, scoring=kaggle_scorer, cv=cvk, n_jobs=-1, random_state=42).fit(R1["Xtr"], R1["ytr"])
    XGB_TUNED, ET_TUNED = xs.best_params_, es.best_params_
print("XGB tuned:", XGB_TUNED); print("ET tuned:", ET_TUNED)

Xtr, Xte, ytr, yte = R1["Xtr"], R1["Xte"], R1["ytr"], R1["yte"]
cands = {
    "Untuned XGBoost (default, 300 trees)": XGBRegressor(n_estimators=300, random_state=42, n_jobs=-1, verbosity=0),
    "Untuned Extra Trees (default, 300 trees)": ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "Tuned XGBoost": XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, **XGB_TUNED),
    "Tuned Extra Trees": ExtraTreesRegressor(random_state=42, n_jobs=-1, **ET_TUNED),
}
tune_rows, tuned_fit = [], {}
for n, m in cands.items():
    tuned_fit[n] = m.fit(Xtr, ytr); tune_rows.append({"model": n, **metrics(yte, m.predict(Xte))})
# [retrospective] blends of the project models with the tuned members
p_x800 = R1["fitted"]["XGBoost"].predict(Xte); p_et2 = R1["fitted"]["Extra Trees"].predict(Xte)
p_ett = tuned_fit["Tuned Extra Trees"].predict(Xte)
tune_rows.append({"model": "Blend XGB + ET (final project model)", **metrics(yte, (p_x800 + p_et2) / 2)})
tune_rows.append({"model": "[retro] Blend XGB + tuned ET", **metrics(yte, (p_x800 + p_ett) / 2)})
tuning = pd.DataFrame(tune_rows).set_index("model").sort_values("R2 (Rs)", ascending=False)
tuning.round(4).to_csv(f"{OUT_DIR}/tuning_holdout.csv"); tuning.round(4)

XGB tuned: {'colsample_bytree': 0.5852620618436457, 'gamma': 0.03252579649263976, 'learning_rate': 0.1433551198429333, 'max_depth': 4, 'min_child_weight': 2, 'n_estimators': 400, 'reg_alpha': 0.015966252220214194, 'reg_lambda': 1.254469128110745, 'subsample': 0.6964101864104046}
ET tuned: {'n_estimators': 700, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': 0.85, 'max_depth': 30, 'bootstrap': True}


,log RMSE,log MAE,R2 (log),R2 (Rs),MAE (Rs),MAPE (%)
model,,,,,,
[retro] Blend XGB + tuned ET,0.1365,0.0821,0.9819,0.9276,3634.4016,8.0926
Blend XGB + ET (final project model),0.1354,0.0800,0.9822,0.9259,3620.5406,7.9163
Tuned Extra Trees,0.1441,0.0861,0.9798,0.9252,3713.9427,8.4475
Tuned XGBoost,0.1522,0.0997,0.9775,0.9220,4133.1994,9.8467
"Untuned XGBoost (default, 300 trees)",0.1451,0.0862,0.9795,0.9170,3900.9977,8.6540
"Untuned Extra Trees (default, 300 trees)",0.1554,0.0847,0.9765,0.9036,3908.0843,8.5334


## G. Retrospective analyses (added during consolidation)

### G.1 Equivalent `ColumnTransformer` pipeline [retrospective]
The project encoded features with `pd.get_dummies`. The same preprocessing expressed as a scikit-learn `ColumnTransformer` handles unseen test categories automatically (`handle_unknown='ignore'`). It is cross-validated on all 14,431 de-duplicated rows, so its scores are not directly comparable with the 11,544-row training-split CV in Part D.

In [25]:
ct_df = fill_duration_round1(train_raw.assign(prefix=train_raw.flight.str.strip().str.upper().str[:2]))
ct_df["route"] = ct_df.source_city + "_to_" + ct_df.destination_city
ct_df = ct_df.drop(columns=["id", "airline", "flight", "source_city", "destination_city"]).drop_duplicates()
ct_X, ct_y = ct_df.drop(columns="price"), np.log1p(ct_df.price)
cat_f = ["departure_time", "arrival_time", "prefix", "route"]
preprocess = ColumnTransformer([
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_f),
    ("ordinal", OrdinalEncoder(categories=[["zero", "one", "two_or_more"], ["Economy", "Business"]]), ["stops", "flight_class"]),
    ("scale", StandardScaler(), ["duration", "days_left"]),
])
ridge_ct = Pipeline([("prep", preprocess), ("scale_all", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
blend_ct = Pipeline([("prep", preprocess), ("model", VotingRegressor([("xgb", R1["models"]["XGBoost"]), ("et", R1["models"]["Extra Trees"])]))])
print("rows:", ct_X.shape, "| features after transform:", preprocess.fit_transform(ct_X).shape[1])
ct_res = pd.DataFrame({n: cv_metrics(p, ct_X, ct_y, cvk) for n, p in [("Ridge (ColumnTransformer)", ridge_ct), ("Blend XGB+ET (ColumnTransformer)", blend_ct)]}).T
ct_res.round(4).to_csv(f"{OUT_DIR}/columntransformer_cv.csv"); ct_res.round(4)

rows: (14431, 8) | features after transform: 65


,log RMSE,log MAE,R2 (log),R2 (Rs),MAE (Rs),MAPE (%),log RMSE sd,MAPE sd
Ridge (ColumnTransformer),0.2500,0.1758,0.9376,0.8388,6377.1821,17.2989,0.0064,0.1669
Blend XGB+ET (ColumnTransformer),0.1326,0.0780,0.9824,0.9301,3370.1783,7.7100,0.0043,0.1146


### G.2 Extra engineered features: log, ratio and interaction [retrospective]

In [26]:
def add_features(X):
    X = X.copy()
    X["log_duration"] = np.log1p(X.duration); X["log_days_left"] = np.log1p(X.days_left)
    X["duration_per_stop"] = X.duration / (X.stops + 1)          # ratio
    X["urgency"] = 1 / X.days_left                               # ratio
    X["class_x_days_left"] = X.flight_class * X.days_left        # interactions
    X["class_x_duration"] = X.flight_class * X.duration
    X["class_x_stops"] = X.flight_class * X.stops
    return X
blend = VotingRegressor([("xgb", R1["models"]["XGBoost"]), ("et", R1["models"]["Extra Trees"])])
fe = pd.DataFrame({"65 base features": cv_metrics(blend, R1["Xtr"], R1["ytr"], cvk),
                   "+ 7 engineered features": cv_metrics(blend, add_features(R1["Xtr"]), R1["ytr"], cvk)}).T
fe.round(4).to_csv(f"{OUT_DIR}/feature_engineering_cv.csv"); fe.round(4)

,log RMSE,log MAE,R2 (log),R2 (Rs),MAE (Rs),MAPE (%),log RMSE sd,MAPE sd
65 base features,0.1352,0.0802,0.9817,0.9343,3341.3528,7.9271,0.0054,0.2756
+ 7 engineered features,0.1364,0.0799,0.9814,0.9314,3382.3070,7.9297,0.0048,0.2783


### G.3 Blend-weight optimisation (the regression analogue of threshold tuning) [retrospective]

In [27]:
w = np.round(np.linspace(0, 1, 21), 2)
sweep = pd.DataFrame([{"w_xgb": a, **metrics(R1["ytr"], a * oof_xgb + (1 - a) * oof_et)} for a in w]).set_index("w_xgb")
best_w = sweep["log RMSE"].idxmin()
print("best XGB weight (OOF log RMSE):", best_w, "| log RMSE", round(sweep.loc[best_w, "log RMSE"], 4), "vs 0.5:", round(sweep.loc[0.5, "log RMSE"], 4))
print("hold-out check — 0.5:", round(metrics(R1["yte"], (p_x800 + p_et2) / 2)["log RMSE"], 4),
      f"| {best_w}:", round(metrics(R1["yte"], best_w * p_x800 + (1 - best_w) * p_et2)["log RMSE"], 4))
fig, ax = plt.subplots(figsize=(7, 3.5)); ax.plot(sweep.index, sweep["log RMSE"], color=BLUE, marker="o", ms=3)
ax.axvline(0.5, color=GREY, ls="--"); ax.axvline(best_w, color=RED, ls=":")
ax.set_xlabel("weight on XGBoost (1 − weight on Extra Trees)"); ax.set_ylabel("OOF log RMSE"); ax.set_title("Blend weight sweep (round-1 training split)")
savefig("fig_blend_weight"); sweep.round(4).to_csv(f"{OUT_DIR}/blend_weight_sweep.csv")

best XGB weight (OOF log RMSE): 0.65 | log RMSE 0.1349 vs 0.5: 0.1353
hold-out check — 0.5: 0.1354 | 0.65: 0.1349


### G.4 Experiments with a 20 % subset of the training data [retrospective]

In [28]:
frac_rows = []
for frac in [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]:
    if frac < 1:
        Xs, _, ys, _ = train_test_split(R1["Xtr"], R1["ytr"], train_size=frac, random_state=42, stratify=R1["Xtr"].flight_class)
    else:
        Xs, ys = R1["Xtr"], R1["ytr"]
    for name in (["Ridge", "Random Forest", "Extra Trees", "XGBoost", "Blend XGB + ET"] if frac == 0.2 else ["Blend XGB + ET"]):
        m = blend if name == "Blend XGB + ET" else R1["models"][name]
        m = clone(m).fit(Xs, ys)
        frac_rows.append({"fraction": frac, "train rows": len(Xs), "model": name, **metrics(R1["yte"], m.predict(R1["Xte"]))})
frac_df = pd.DataFrame(frac_rows); frac_df.round(4).to_csv(f"{OUT_DIR}/subset_experiments.csv", index=False)
display(frac_df[frac_df.fraction == 0.2].round(4))
lc = frac_df[frac_df.model == "Blend XGB + ET"]
fig, ax = plt.subplots(figsize=(7, 3.5)); ax.plot(lc["train rows"], lc["log RMSE"], marker="o", color=BLUE)
for _, r in lc.iterrows(): ax.annotate(f"{int(r.fraction*100)}%", (r["train rows"], r["log RMSE"]), textcoords="offset points", xytext=(4, 6), fontsize=8)
ax.set_xlabel("training rows"); ax.set_ylabel("hold-out log RMSE"); ax.set_title("Learning curve of the XGB + ET blend"); savefig("fig_learning_curve")
lc.round(4)

,fraction,train rows,model,log RMSE,log MAE,R2 (log),R2 (Rs),MAE (Rs),MAPE (%)
1,0.2,2308,Ridge,0.2531,0.1744,0.9376,0.8307,6614.1398,17.0915
2,0.2,2308,Random Forest,0.1958,0.1122,0.9627,0.8742,4863.8323,11.0699
3,0.2,2308,Extra Trees,0.1915,0.1073,0.9643,0.8754,4721.4729,10.5564
4,0.2,2308,XGBoost,0.1860,0.1086,0.9663,0.8794,4690.3668,10.8396
5,0.2,2308,Blend XGB + ET,0.1823,0.1046,0.9676,0.8824,4580.3868,10.3040


,fraction,train rows,model,log RMSE,log MAE,R2 (log),R2 (Rs),MAE (Rs),MAPE (%)
0,0.1,1154,Blend XGB + ET,0.2025,0.1188,0.9601,0.8691,5196.2735,11.7299
5,0.2,2308,Blend XGB + ET,0.1823,0.1046,0.9676,0.8824,4580.3868,10.3040
6,0.4,4617,Blend XGB + ET,0.1547,0.0910,0.9767,0.9151,3946.0834,8.9177
7,0.6,6926,Blend XGB + ET,0.1431,0.0850,0.9801,0.9246,3686.2404,8.4191
8,0.8,9235,Blend XGB + ET,0.1381,0.0825,0.9814,0.9248,3684.9385,8.1929
9,1.0,11544,Blend XGB + ET,0.1354,0.0800,0.9822,0.9259,3620.5406,7.9163


### G.5 Unsupervised learning: PCA and k-means [retrospective]

In [29]:
Xs = StandardScaler().fit_transform(X1_full)
pca = PCA().fit(Xs); cum = np.cumsum(pca.explained_variance_ratio_)
print("components for 80% / 90% / 95% variance:", [int(np.searchsorted(cum, t) + 1) for t in (0.8, 0.9, 0.95)])
Z = PCA(n_components=2, random_state=42).fit_transform(Xs)
ks, inertia, sil = range(2, 11), [], []
samp = np.random.default_rng(42).choice(len(Xs), 4000, replace=False)
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs); inertia.append(km.inertia_)
    sil.append(silhouette_score(Xs[samp], km.labels_[samp]))
unsup = pd.DataFrame({"k": list(ks), "inertia": inertia, "silhouette": sil}); unsup.round(4).to_csv(f"{OUT_DIR}/kmeans_scan.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(range(1, len(cum) + 1), cum, color=BLUE); axes[0].axhline(0.9, color=GREY, ls="--"); axes[0].set_xlabel("components"); axes[0].set_ylabel("cumulative explained variance"); axes[0].set_title("PCA on 65 scaled features")
sc_ = axes[1].scatter(Z[:, 0], Z[:, 1], c=y1_full, s=3, cmap="viridis"); plt.colorbar(sc_, ax=axes[1], label="log1p(price)"); axes[1].set_title("First two principal components")
axes[2].plot(unsup.k, unsup.silhouette, marker="o", color=BLUE); axes[2].set_xlabel("k"); axes[2].set_ylabel("silhouette (4,000-row sample)"); axes[2].set_title("k-means: silhouette by k")
savefig("fig_unsupervised"); unsup.round(4)

components for 80% / 90% / 95% variance: [42, 49, 53]


,k,inertia,silhouette
0,2,900403.9364,0.1438
1,3,877024.4652,0.0321
2,4,859421.1701,0.0242
3,5,845327.0450,0.0272
4,6,822254.8909,0.0421
5,7,807343.7955,0.0521
6,8,788052.9656,0.0645
7,9,780469.4230,0.0598
8,10,768199.8135,0.0648


In [30]:
k_best = int(unsup.loc[unsup.silhouette.idxmax(), "k"])
km = KMeans(n_clusters=k_best, n_init=10, random_state=42).fit(Xs)
prof = pd.DataFrame({"cluster": km.labels_, "price": np.expm1(y1_full), "business": X1_full.flight_class, "stops": X1_full.stops,
                     "duration": X1_full.duration, "days_left": X1_full.days_left}).groupby("cluster").agg(
    rows=("price", "size"), median_price=("price", "median"), business_share=("business", "mean"),
    mean_stops=("stops", "mean"), mean_duration=("duration", "mean"), mean_days_left=("days_left", "mean"))
print("k with best silhouette:", k_best); prof.round(2).to_csv(f"{OUT_DIR}/kmeans_profiles.csv"); prof.round(2)

k with best silhouette: 2


,rows,median_price,business_share,mean_stops,mean_duration,mean_days_left
cluster,,,,,,
0,10517,8817.0,0.24,0.05,2.48,22.28
1,3914,46420.0,0.87,1.02,9.39,21.96


In [31]:
# Does adding the k-means cluster id as a feature help the blend? [retrospective]
km2 = KMeans(n_clusters=k_best, n_init=10, random_state=42)
sc2 = StandardScaler().fit(R1["Xtr"])
Xtr_c = R1["Xtr"].assign(cluster=km2.fit_predict(sc2.transform(R1["Xtr"])))
clu = pd.DataFrame({"base": cv_metrics(blend, R1["Xtr"], R1["ytr"], cvk), "+ cluster id": cv_metrics(blend, Xtr_c, R1["ytr"], cvk)}).T
clu.round(4).to_csv(f"{OUT_DIR}/cluster_feature_cv.csv"); clu.round(4)

,log RMSE,log MAE,R2 (log),R2 (Rs),MAE (Rs),MAPE (%),log RMSE sd,MAPE sd
base,0.1352,0.0802,0.9817,0.9343,3341.3528,7.9271,0.0054,0.2756
+ cluster id,0.1357,0.0801,0.9815,0.9327,3362.7171,7.9093,0.0058,0.2700


## H. Final model: XGB + ET blend on all round-1 training rows → predictions for `test.csv`

In [32]:
final = VotingRegressor([("xgb", R1["models"]["XGBoost"]), ("et", R1["models"]["Extra Trees"])]).fit(X1_full, y1_full)
pred_test = np.expm1(final.predict(X1_kaggle))
sub = pd.DataFrame({"id": test_raw.id, "price": pred_test}); sub.to_csv("submission.csv", index=False)
print(sub.shape); print(sub.price.describe().round(0))

(3898, 2)
count      3898.0
mean      29371.0
std       37656.0
min        3850.0
25%        7254.0
50%       10025.0
75%       35685.0
max      256703.0
Name: price, dtype: float64


In [33]:
# Figure: predicted test prices vs training prices, by class
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for i, cls in enumerate(["Economy", "Business"]):
    tr_ = train_raw.price[train_raw.flight_class == cls]; te_ = sub.price[test_raw.flight_class.values == cls]
    bins = np.logspace(np.log10(min(tr_.min(), te_.min())), np.log10(max(tr_.max(), te_.max())), 40)
    ax[i].hist(tr_, bins=bins, density=True, alpha=0.55, color=GREY, label=f"train actual (n={len(tr_)})")
    ax[i].hist(te_, bins=bins, density=True, alpha=0.55, color=BLUE, label=f"test predicted (n={len(te_)})")
    ax[i].set_xscale("log"); ax[i].set_title(cls); ax[i].set_xlabel("price (Rs, log)"); ax[i].legend(fontsize=8)
savefig("fig_test_predictions")
print(pd.DataFrame({"train median": train_raw.groupby("flight_class").price.median(),
                    "test predicted median": sub.groupby(test_raw.flight_class.values).price.median()}).round(0))

          train median  test predicted median
Business       38546.0                39072.0
Economy         7587.0                 7740.0


In [34]:
# Figure: test-set predictions by days_left and by duration (model behaviour on unseen rows)
tv = test_raw.assign(pred=sub.price.values)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for cls, col in [("Economy", GREY), ("Business", BLUE)]:
    g = tv[tv.flight_class == cls].groupby("days_left").pred.median(); ax[0].plot(g.index, g.values, color=col, label=f"{cls} (test, predicted)")
    g2 = train_raw[train_raw.flight_class == cls].groupby("days_left").price.median(); ax[0].plot(g2.index, g2.values, color=col, ls=":", label=f"{cls} (train, actual)")
ax[0].set_yscale("log"); ax[0].set_xlabel("days left"); ax[0].set_ylabel("median price (Rs)"); ax[0].legend(fontsize=7); ax[0].set_title("Test predictions follow the training price curve")
imp = pd.DataFrame({"XGBoost (gain share)": pd.Series(final.named_estimators_["xgb"].get_booster().get_score(importance_type="gain")),
                    "Extra Trees (impurity)": pd.Series(final.named_estimators_["et"].feature_importances_, index=X1_full.columns)}).fillna(0)
imp["XGBoost (gain share)"] /= imp["XGBoost (gain share)"].sum()
top = imp.sort_values("Extra Trees (impurity)", ascending=False).head(12).iloc[::-1]
top.plot.barh(ax=ax[1], color=[GREY, BLUE]); ax[1].set_xlabel("importance share"); ax[1].set_title("Top-12 features (final blend members)")
savefig("fig_test_by_days_and_importance")
imp.sort_values("Extra Trees (impurity)", ascending=False).round(4).to_csv(f"{OUT_DIR}/feature_importance.csv"); top.iloc[::-1].round(4)

,XGBoost (gain share),Extra Trees (impurity)
flight_class,0.5882,0.7799
duration,0.0276,0.0545
prefix_6E,0.0233,0.0383
stops,0.0117,0.0199
prefix_EK,0.0281,0.0169
prefix_EY,0.0157,0.0141
prefix_UL,0.0478,0.0125
days_left,0.0016,0.0120
prefix_AI,0.0305,0.0073
route_Metropolis_to_Night City,0.0140,0.0047


## Summary
* Best Kaggle public score: **0.96238** — XGB + ET blend on round-1 data (`submission.csv` reproduces this model on the full training set).
* Round-2 preprocessing gave a slightly better cross-validated log RMSE but a lower Kaggle score (0.96077).
* Tuning improved Extra Trees on the raw-price R² hold-out; XGBoost tuning was inconclusive.
* Retrospective analyses (ColumnTransformer, extra features, blend weights, 20 % subset, PCA / k-means) are reported in Section G.